# 🔁 Cuaderno 04 — Matrices especiales y métodos iterativos

**Métodos Numéricos · Unidad 3: Sistemas de Ecuaciones Lineales**

Este cuaderno es **taller, no libro**. Tiene dos mitades bien distintas, y el
orden importa:

1. **Matrices especiales** (secciones 2 a 5): Thomas y Cholesky. Son métodos
   **exactos**, como los de los cuadernos 02 y 03, y aprovechan que la matriz
   tiene ceros en casi todos lados. Aquí se sigue trabajando con `M.sym`.
2. **Métodos iterativos** (secciones 6 a 9): Jacobi y Gauss-Seidel. Son de
   **punto flotante** —y aquí sí, a propósito— porque se usan en sistemas
   grandes, donde las fracciones exactas crecen sin control.

📖 Teoría completa:
[`03_metodos_iterativos.md`](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/master/docs/metodos/03_metodos_iterativos.md) ·
[`04_matrices_especiales.md`](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/master/docs/metodos/04_matrices_especiales.md)

### 🎯 Lo que vas a entregar

| Pregunta | Qué |
|---|---|
| **9** | Una barrida de **Jacobi a mano** (con el vector viejo) |
| **10** | Una barrida de **Gauss-Seidel a mano** (reutilizando lo nuevo) |
| **11** | `mi_jacobi(A, b, tol, max_iter)` — **el método completo** |
| **12** | `mi_gauss_seidel(A, b, tol, max_iter)` — **el método completo** |
| **13** | `mi_thomas(e, f, g, r)` — el sistema tridiagonal |

### ⚙️ Dos aritméticas en un mismo cuaderno

- **Exacta** (`M.sym`) para Thomas, Cholesky y el determinante: verás `3/2`, no
  `1.5000000000000002`.
- **Decimal** (`float`) para Jacobi y Gauss-Seidel: es como se usan de verdad, y
  así el error de redondeo que discute la teoría **sí aparece**.


In [1]:
#@title ⚙️ Cargar entorno estilo MATLAB (ejecuta PRIMERO)

import os, sys, subprocess, importlib

# --- Detecta si corremos en Google Colab ---
try:
    import google.colab
    ES_COLAB = True
except ImportError:
    ES_COLAB = False

REPO_URL = "https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales.git"


def _buscar_raiz(base, saltos=6):
    """Sube desde `base` hasta la carpeta que tiene config/config.yaml."""
    b = base
    for _ in range(saltos):
        if os.path.exists(os.path.join(b, "config", "config.yaml")):
            return os.path.abspath(b)
        padre = os.path.dirname(b)
        if padre == b:
            break
        b = padre
    return None


if ES_COLAB:
    # ============================================================
    # Google Colab: montar Drive y traer el repositorio
    # ============================================================
    # El repo se clona DENTRO de tu Drive, no en /content (que se borra al
    # cerrar la sesión). Así sobrevive entre sesiones y `git pull` solo
    # actualiza lo que cambió.
    from google.colab import drive

    drive.mount("/content/drive")

    BASE = "/content/drive/MyDrive/CursoMN"
    REPO = os.path.join(BASE, "U3_Repo")
    os.makedirs(BASE, exist_ok=True)

    try:
        if not os.path.exists(os.path.join(REPO, ".git")):
            print("Clonando el repositorio (la primera vez tarda un poco)...")
            subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
        else:
            print("Actualizando el repositorio...")
            subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
        print("✅ Repositorio listo en", REPO)
    except Exception as _exc:
        # Lo más común: el alumno guardó el cuaderno DENTRO del clon (Colab lo
        # reescribe), así que git se niega a sobrescribirlo y el pull aborta.
        # Aquel alumno se quedaba sin la carpeta grader/, que es lo único
        # imprescindible. Aquí se fuerza sólo eso, sin tocar sus notebooks.
        print("⚠️ No pude actualizar el repositorio completo:", _exc)
        print("   Fuerzo solo lo que el cuaderno necesita (tus notebooks no se tocan)...")
        try:
            subprocess.run(["git", "-C", REPO, "fetch", "origin", "master"],
                           check=True)
            subprocess.run(["git", "-C", REPO, "checkout", "FETCH_HEAD", "--",
                            "grader", "lib/robomat.py"], check=True)
            print("✅ grader/ y lib/robomat.py quedaron al día.")
        except Exception as _exc2:
            print("⛔ Tampoco pude actualizar solo esos archivos:", _exc2)
            print(f"   Corre esto en una celda y vuelve a ejecutar la carga:")
            print(f"   !git -C '{REPO}' fetch origin master && git -C '{REPO}' checkout FETCH_HEAD -- grader lib/robomat.py")
else:
    # En tu computadora el repo ya está en disco: solo hay que encontrarlo.
    _ruta_nb = globals().get("__vsc_ipynb_file__")      # VS Code
    REPO = _buscar_raiz(os.path.dirname(os.path.abspath(_ruta_nb))) if _ruta_nb else None
    REPO = REPO or _buscar_raiz(os.getcwd())

if REPO is None:
    raise RuntimeError(
        "No encuentro la carpeta del repositorio (la que tiene config/config.yaml).\n"
        "Abre el cuaderno desde dentro de U3_Repo o revisa que el clon de Colab exista."
    )

print("Repositorio :", REPO)

# ============================================================
# Biblioteca del curso (lib/robomat.py)
# ============================================================
# En sys.path van las DOS raíces que hacen falta:
#   REPO/lib    -> robomat.py y el paquete mn_motor/
#   REPO        -> el paquete preguntas/ (las preguntas de este cuaderno)
for _ruta in (os.path.join(REPO, "lib"), REPO):
    if _ruta not in sys.path:
        sys.path.insert(0, _ruta)

import robomat as _robomat
importlib.reload(_robomat)          # recoge los cambios si editaste la libreria
from robomat import *               # noqa: F401,F403

print("✅ Entorno cargado desde:", _robomat.__file__)


# ============================================================
# Motor de calificación
# ============================================================
# En Colab SIEMPRE se usa el blob ofuscado. En tu computadora puedes escoger
# "modulos": importa lib/mn_motor/ tal cual, y así funcionan F12, el
# autocompletado y los puntos de ruptura. Es el modo para depurar.
MODO_MOTOR = "modulos"

MODO = "blob" if ES_COLAB else MODO_MOTOR

if MODO == "modulos":
    try:
        # Se borran del caché para que una edición en lib/mn_motor/ o en
        # preguntas/ se note al volver a correr la celda, sin reiniciar.
        for _n in [n for n in sys.modules
                   if n.split(".")[0] in ("mn_motor", "preguntas")]:
            del sys.modules[_n]
        # Import ESTÁTICO después del borrado: es lo que Pylance analiza.
        import mn_motor as _motor
        from mn_motor import *        # noqa: F401,F403
        if "usar_cuaderno" not in globals():
            # Un clon a medias deja lib/mn_motor/ como carpeta vacía: el import
            # "funciona" pero no trae nada, y eso NO es un ImportError. Sin
            # esto, el cuaderno moría más abajo con un NameError críptico.
            raise ImportError("lib/mn_motor está incompleto (no trae usar_cuaderno)")
    except ImportError as _exc:
        # En un clon del alumno no existe preguntas/: no hay clave de
        # respuestas. Sin problema: el blob trae lo mismo.
        print("ℹ️ No pude importar el motor del repositorio:", _exc)
        print("   Uso el motor ofuscado del blob.")
        MODO = "blob"

if MODO == "blob":
    import base64 as _b64
    import zlib as _zlib

    # Cada cuaderno tiene su propio blob: aquí va el del Cuaderno 04.
    ruta_motor = os.path.join(REPO, "grader", "u3_04_especiales_iterativos.txt")
    if not os.path.exists(ruta_motor):
        raise FileNotFoundError(
            f"No encuentro {ruta_motor}: tu copia del repositorio está incompleta."
            f" Corre esto en una celda y vuelve a ejecutar:"
            f" !git -C '{REPO}' fetch origin master && git -C '{REPO}'"
            f" checkout FETCH_HEAD -- grader lib/robomat.py")
    with open(ruta_motor, encoding="utf-8") as _f:
        _blob = _f.read().strip()
    _codigo = _zlib.decompress(_b64.b64decode(_blob)).decode("utf-8")
    # _REPO_U3 le dice al motor dónde está el repo: dentro del blob no existe
    # __file__, así que no puede deducirlo solo.
    os.environ["U3_REPO"] = REPO
    _MOTOR = {"__name__": "blob_u3"}
    exec(_codigo, _MOTOR)
    globals().update({_n: _MOTOR[_n] for _n in _MOTOR["__all__"]})


# Este cuaderno califica SU conjunto de preguntas.
# En modo blob ya viene incrustado, pero en modo módulos hay que declararlo.
usar_cuaderno("u3_04_especiales_iterativos", "U3_04_Especiales_Iterativos")

print(f"✅ Motor de calificación cargado ({MODO}) · cuaderno {cuaderno()}")
print("   Sigue con la celda de identificación.")


MessageError: Error: credential propagation was unsuccessful

## 0 · Tu identificación

Todo lo que sigue usa **tus** datos. Al ejecutar la celda de abajo, Colab te
pide permiso para leer tu correo; de ahí salen tu número de control y tu nombre.
Con eso se siembran los números de tus preguntas: **cada alumno recibe datos
distintos**.

Fuera de Colab no hay sesión de Google, así que el correo se escribe a mano en
la celda.


In [ ]:
#@title 👤 Identificación automática (correo institucional)
# ============================================================
# Uso el correo con el que montaste Drive: de ahí salen tu NC y tu nombre.
# ============================================================
if ES_COLAB:
    import requests
    import google.auth
    import google.auth.transport.requests
    from google.colab import auth

    try:
        auth.authenticate_user()
        creds, _ = google.auth.default()
        creds.refresh(google.auth.transport.requests.Request())
        info = requests.get(
            "https://www.googleapis.com/drive/v3/about?fields=user",
            headers={"Authorization": "Bearer " + creds.token},
            timeout=20,
        ).json()
        cuenta = info.get("user", {})
        email = cuenta.get("emailAddress", "")
        nombre = cuenta.get("displayName", "")
    except Exception as e:
        print("⚠️ No pude obtener tu correo automáticamente.")
        print("   Vuelve a ejecutar y acepta el permiso en la ventana de Google.")
        print("   Detalle:", e)
        email, nombre = "", ""
else:
    # Modo local: aquí no hay sesión de Google, escribe tu correo.
    email = "m16330887@hermosillo.tecnm.mx"      # <-- cambia por el tuyo
    nombre = ""                                 # Google solo lo da en Colab

if email and email.lower().endswith(".tecnm.mx"):
    alumno_id = email
    print("✅ Identificado:", email)
    print("   Número de control:", extraer_nc(email))
    print("   Nombre:", nombre if nombre else "(no disponible fuera de Colab)")
else:
    from IPython.display import HTML, display as _display

    alumno_id = None
    _display(HTML(
        '<p style="color:#b00020;background:#ffe0e0;padding:12px;border-radius:8px;'
        'font-weight:bold">⛔ Necesito tu correo institucional (@….tecnm.mx).<br>'
        'Si iniciaste sesión con otra cuenta, ciérrala o usa una ventana de '
        'incógnito.</p>'))
    print("Correo detectado:", repr(email))

if alumno_id is None:
    raise RuntimeError("Detente: sin correo institucional no puedo generar tu tarea.")

# La tarea se genera UNA sola vez: todas las celdas de preguntas llaman a
# pregunta(n) sobre este MI_TAREA, así que todas hablan de lo mismo.
MI_TAREA = generar_tarea(alumno_id, nombre)
print(f"\nListo: {len(MI_TAREA['preguntas'])} preguntas para el NC {MI_TAREA['nc']}.")


⚠️ No pude obtener tu correo automáticamente.
   Vuelve a ejecutar y acepta el permiso en la ventana de Google.
   Detalle: Error: credential propagation was unsuccessful


Correo detectado: ''


RuntimeError: Detente: sin correo institucional no puedo generar tu tarea.

## 1 · El mapa del cuaderno

Los dos primeros métodos aprovechan la **estructura** de la matriz; los dos
últimos renuncian a transformarla y se acercan a la solución paso a paso.

| | Cómo trabaja | Costo | Aritmética |
|---|---|---|---|
| **Thomas** | eliminación sobre una tridiagonal, nunca guarda los ceros | $O(n)$ | exacta |
| **Cholesky** | factoriza $\mathbf{A}=\mathbf{L}\mathbf{L}^{T}$ (simétrica definida positiva) | $\tfrac{1}{3}n^3$ | exacta |
| **Jacobi** | itera usando el vector viejo completo | $O(n^2)$ por iteración | decimal |
| **Gauss-Seidel** | itera reutilizando lo recién calculado | $O(n^2)$ por iteración | decimal |

> 💡 **Por qué los iterativos van en decimal.** No es por comodidad: con
> fracciones exactas, un caso que necesita 47 iteraciones termina con
> denominadores de 33 dígitos. Y peor: en sistemas grandes el número de
> iteraciones depende de detalles de redondeo, que es justamente lo que la
> teoría de la [sección 3.6](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/master/docs/metodos/03_metodos_iterativos.md#sec-3-6)
> discute.


## 2 · Tridiagonales: el algoritmo de Thomas

Una matriz **tridiagonal** solo tiene coeficientes en la diagonal ($f$) y en las
dos diagonales vecinas ($e$ abajo, $g$ arriba):

$$\begin{bmatrix} f_1 & g_1 & 0 & 0 \\ e_2 & f_2 & g_2 & 0 \\ 0 & e_3 & f_3 & g_3 \\ 0 & 0 & e_4 & f_4 \end{bmatrix}$$

Thomas es la eliminación gaussiana **sin pivoteo** sobre esa estructura, en tres
fases:

```
# fase 1: modificar f  (k = 2..n)
e'[k] = e[k] / f'[k-1]
f'[k] = f[k] - e'[k] * g[k-1]

# fase 2: modificar r  (k = 2..n)
r'[k] = r[k] - e'[k] * r'[k-1]

# fase 3: sustitución hacia atrás  (k = n..1)
x[n] = r'[n] / f'[n]
x[k] = (r'[k] - g[k] * x[k+1]) / f'[k]
```

En la **primera** fila no hay nada que modificar: `f'[1] = f[1]` y `r'[1] = r[1]`.
Ese es el error más común al programarlo.

**Convención de los vectores** (la del documento y la que usa la pregunta 13):
los cuatro vectores miden $n$, y las posiciones que no existen valen 0.
`e[i]` multiplica a $x_{i-1}$, así que `e[0]` no se usa; `g[i]` multiplica a
$x_{i+1}$, así que `g[n-1]` no se usa.

> 💡 La ganancia: $O(n)$ en tiempo y memoria, contra $O(n^3)$ y $O(n^2)$ de la
> eliminación normal. No guarda ni un cero.


In [ ]:
#@title 🧮 Thomas a mano: las tres fases con números
# La tridiagonal del documento. La solución es [1, 1, 1].
# OJO: aquí NO hay función. Son las cuentas escritas una por una, con los
# índices fijos de este ejemplo; generalizarlas a cualquier n es tu tarea.
A_tri = M.sym[[2, -1, 0], [-1, 2, -1], [0, -1, 2]]
mostrar(A_tri)

f1, f2, f3 = A_tri[0, 0], A_tri[1, 1], A_tri[2, 2]
e2, e3 = A_tri[1, 0], A_tri[2, 1]
g1, g2 = A_tri[0, 1], A_tri[1, 2]
r = M.sym[[1], [0], [1]]
r1, r2, r3 = r[0, 0], r[1, 0], r[2, 0]
print("f =", [int(v) for v in (f1, f2, f3)],
      "· e =", [0, int(e2), int(e3)],
      "· g =", [int(g1), int(g2), 0])

# --- fase 1 y 2, renglón 2 ---
print("\nrenglón 2:")
e2p = e2 / f1
f2p = f2 - e2p * g1
r2p = r2 - e2p * r1
print(f"  e'2 = e2 / f1       = {e2} / {f1} = {e2p}")
print(f"  f'2 = f2 - e'2·g1   = {f2} - ({e2p})({g1}) = {f2p}")
print(f"  r'2 = r2 - e'2·r1   = {r2} - ({e2p})({r1}) = {r2p}")

# --- fase 1 y 2, renglón 3 ---
print("renglón 3:")
e3p = e3 / f2p
f3p = f3 - e3p * g2
r3p = r3 - e3p * r2p
print(f"  e'3 = e3 / f'2      = {e3} / {f2p} = {e3p}")
print(f"  f'3 = f3 - e'3·g2   = {f3} - ({e3p})({g2}) = {f3p}")
print(f"  r'3 = r3 - e'3·r'2  = {r3} - ({e3p})({r2p}) = {r3p}")

# --- fase 3: de abajo hacia arriba ---
print("\nsustitución hacia atrás:")
x3 = r3p / f3p
x2 = (r2p - g2 * x3) / f2p
x1 = (r1 - g1 * x2) / f1
print(f"  x3 = r'3 / f'3          = {r3p} / {f3p} = {x3}")
print(f"  x2 = (r'2 - g2·x3) / f'2 = ({r2p} - ({g2})({x3})) / {f2p} = {x2}")
print(f"  x1 = (r1 - g1·x2) / f1   = ({r1} - ({g1})({x2})) / {f1} = {x1}")
mostrar(M.sym[[x1, x2, x3]].T)
print("de regalo, el determinante: f'1 · f'2 · f'3 =", f1 * f2p * f3p)


## 3 · Tu turno: el algoritmo de Thomas

Escribe `mi_thomas(e, f, g, r)` en la celda de abajo. Devuelve $\mathbf{x}$ como
**columna**, y aquí sí se usa aritmética exacta (es un método directo).

`practica('prog_thomas')` lo prueba con el ejemplo del documento. No gasta
intentos.

> ✍️ **La celda anterior no es una función.** Es la cuenta hecha con los números de este ejemplo. Generalizarla —con ciclos y para cualquier $n$— es lo que tienes que escribir tú.


In [ ]:
#@title ✍️ Escribe el algoritmo de Thomas
def mi_thomas(e, f, g, r):
    """Resuelve el sistema tridiagonal y devuelve x como columna."""
    raise NotImplementedError("Traduce aquí las tres fases de la sección 2")


practica('prog_thomas')


In [ ]:
#@title 📝 Pregunta 13 (vale 1 punto)
pregunta(13)


## 4 · Cholesky: una sola triangular

Si $\mathbf{A}$ es **simétrica** ($a_{ij} = a_{ji}$) y **definida positiva**
($\mathbf{x}^{T}\mathbf{A}\mathbf{x} > 0$ para todo $\mathbf{x}\neq\mathbf{0}$),
la factorización se reduce a una sola matriz:

$$\mathbf{A} = \mathbf{L}\mathbf{L}^{T}$$

Las recurrencias, renglón por renglón, son

$$l_{kk} = \sqrt{a_{kk} - \sum_{j<k} l_{kj}^{2}}, \qquad l_{ki} = \frac{a_{ki} - \sum_{j<i} l_{ij}l_{kj}}{l_{ii}}$$

Se calcula primero la **raíz de la diagonal** y luego se dividen las entradas del
renglón. Tres consecuencias: cuesta $\tfrac{1}{3}n^3$ (la mitad que LU), solo se
guarda la mitad inferior y **no necesita pivoteo**, porque el radicando siempre
es positivo.

La celda siguiente la calcula con la matriz del documento y comprueba que
$\mathbf{L}\mathbf{L}^{T} = \mathbf{A}$.


In [ ]:
#@title 🧪 Cholesky sobre la matriz del documento
Ac = M.sym[[4, 12, -16], [12, 37, -43], [-16, -43, 98]]
mostrar(Ac)
print("¿simétrica?", Ac == Ac.T)

n = Ac.rows
Lc = zeros_sym(n, n)
for k in range(n):
    radicando = Ac[k, k] - sum(Lc[k, j] ** 2 for j in range(k))
    Lc[k, k] = sqrt(radicando)              # sqrt exacta: deja sqrt(2) si hace falta
    for i in range(k + 1, n):
        Lc[i, k] = (Ac[i, k] - sum(Lc[i, j] * Lc[k, j] for j in range(k))) / Lc[k, k]

print("L =")
mostrar(Lc)
print("¿L @ Lᵀ = A?", (Lc @ Lc.T) == Ac)
print("¿exacto?", es_exacto(Lc))


## 5 · ¿Y el determinante?

El determinante dice si el sistema tiene solución única: si $\det(\mathbf{A}) = 0$
la matriz es **singular** y no hay solución única. Calcularlo por cofactores
cuesta $O(n!)$, así que nunca se hace así: se saca **de regalo** de la
factorización que ya se hizo.

**La regla general** (repaso de los cuadernos 02 y 03). Restar un múltiplo de un
renglón no cambia el determinante, e intercambiar dos renglones le cambia el
signo. Como la eliminación termina en una triangular y el determinante de una
triangular es el producto de su diagonal,

$$\det(\mathbf{A}) = (-1)^{p}\,u_{11}\,u_{22}\cdots u_{nn} = (-1)^{p}\prod_{k=1}^{n} u_{kk}$$

donde $p$ es el número de intercambios. Con LU se ve mejor: $\det(\mathbf{A}) = \det(\mathbf{L})\det(\mathbf{U})$ y en Doolittle $\det(\mathbf{L}) = 1$.

**En una tridiagonal**, Thomas es una factorización LU con unos en la diagonal de
$\mathbf{L}$, así que

$$\det(\mathbf{A}) = f'_1\,f'_2\cdots f'_n = \prod_{k=1}^{n} f'_k.$$

**En Cholesky**, como $\mathbf{A} = \mathbf{L}\mathbf{L}^{T}$,

$$\det(\mathbf{A}) = \left(l_{11}\,l_{22}\cdots l_{nn}\right)^{2} = \left(\prod_{k=1}^{n} l_{kk}\right)^{2},$$

que sale siempre positivo: otra manera de ver que una matriz definida positiva no
puede ser singular.

> 📐 **La productoria** $\prod$ es al producto lo que $\sum$ es a la suma: multiplicar en vez de sumar. Se escribe con un índice abajo y otro arriba, y dice *desde dónde hasta dónde* se multiplica. Por eso $\prod_{k=1}^{n} u_{kk}$ es simplemente el producto de la diagonal, escrito en corto. En esta unidad conviene saber leer las dos formas: la desarrollada deja ver el resultado, la compacta es la que aparece en los libros.

> 💡 El costo extra es **cero**: son números que ya estaban calculados.
>
> ⚠️ Con $n$ grande el determinante se desborda (para $n=100$ puede ser del orden
> de $10^{200}$) y solo quedan confiables el signo y el orden de magnitud. Para
> decidir si el sistema es singular conviene mirar el número de condición
> (Cuaderno 03, sección 7) antes que el determinante.


In [ ]:
#@title 🧮 Los tres determinantes, y de dónde sale cada uno
# --- 1) Tridiagonal: la diagonal de U son los f' de Thomas ---
A_tri = M.sym[[2, -1, 0], [-1, 2, -1], [0, -1, 2]]
mostrar(A_tri)

U_tri = M.sym[A_tri.tolist()]
for k in range(2):
    m = U_tri[k + 1, k] / U_tri[k, k]
    U_tri[k + 1, :] = U_tri[k + 1, :] - m * U_tri[k, :]

f_prima = [U_tri[i, i] for i in range(3)]
print("pivotes modificados (= los f' de Thomas):", f_prima)
print("su producto    :", f_prima[0] * f_prima[1] * f_prima[2])
print("det exacto     :", A_tri.det())

# --- 2) Cholesky: el cuadrado del producto de la diagonal de L ---
diagonal_L = [Lc[i, i] for i in range(3)]
producto = diagonal_L[0] * diagonal_L[1] * diagonal_L[2]
print("\ndiagonal de L  :", diagonal_L)
print("su producto^2  :", producto ** 2)
print("det exacto     :", Ac.det())

# --- 3) el repaso general, con la matriz del Cuaderno 02 ---
A2 = M.sym[[2, 1, -1], [-3, -1, 2], [-2, 1, 2]]
U2 = M.sym[A2.tolist()]
for k in range(2):
    for i in range(k + 1, 3):
        m = U2[i, k] / U2[k, k]
        U2[i, :] = U2[i, :] - m * U2[k, :]

pivotes = [U2[i, i] for i in range(3)]
print("\npivotes        :", pivotes)
print("producto       :", pivotes[0] * pivotes[1] * pivotes[2],
      "(sin intercambios, p = 0)")
print("det exacto     :", A2.det())


## 6 · Métodos iterativos: qué cambia

Hasta aquí todo era **directo**: se transforma la matriz y la solución sale en un
número finito y predecible de pasos. Un método **iterativo** hace otra cosa:
parte de un vector inicial $\mathbf{x}^{(0)}$ y genera

$$\mathbf{x}^{(0)},\ \mathbf{x}^{(1)},\ \mathbf{x}^{(2)},\ \dots$$

despejando de cada ecuación **su propia** incógnita:

$$x_i = \frac{b_i - \sum_{j \neq i} a_{ij}\,x_j}{a_{ii}}$$

La ventaja aparece con matrices **grandes y dispersas**, donde eliminar gastaría
memoria y tiempo en ceros. La desventaja: la convergencia no está garantizada.

### Aquí se trabaja en decimal

Con fracciones exactas estos métodos no tienen sentido práctico: 47 iteraciones
dejan denominadores de 33 dígitos. Además el error de redondeo **es parte de la
lección** (sección 3.6 del documento). Así que en todo lo que sigue se usa
`float`, y por eso los números salen como `1.9999999` y no como `2`.

### El problema del arranque: hay que reordenar

Antes de iterar hay que asegurar la **dominancia diagonal estricta**:

$$|a_{ii}| > \sum_{j \neq i} |a_{ij}| \qquad \forall i$$

Si no se cumple, una solución barata es **cambiar el orden de las ecuaciones**
(no de las incógnitas). La regla que se usa en este cuaderno: *para cada columna
$k$, de las ecuaciones que aún no se han usado, tomar la del mayor $|a_{ik}|$*.

La celda siguiente muestra los dos efectos: primero verifica la dominancia,
después reordena, y al final enseña qué pasa si uno se salta ese paso.

> ✍️ **La celda anterior no es una función.** Es la cuenta hecha con los números de este ejemplo. Generalizarla —con ciclos y para cualquier $n$— es lo que tienes que escribir tú.


In [ ]:
#@title 🔀 Por qué hay que reordenar (y qué pasa si no)
# Tampoco hay funciones aquí: primero se revisa la dominancia a mano, después
# se ve el desastre de iterar sin reordenar.
A_des = [[2, 7, 1],
         [1, -3, 12],
         [4, 1, -1]]
b_des = [19, 31, 3]

print("Sistema TAL COMO VIENE (las filas están desordenadas):")
for i in range(3):
    resto = sum(abs(A_des[i][j]) for j in range(3) if j != i)
    print(f"  renglón {i + 1}: |a_ii| = |{A_des[i][i]}|  vs  suma de los demás = {resto}"
          f"   -> dominante: {abs(A_des[i][i]) > resto}")
print("los tres renglones fallan: así no hay ninguna garantía de convergencia")

# --- el reordenamiento, a mano, columna por columna ---
print("\nSe reordena eligiendo, para cada columna, la ecuación del mayor |a_ik|:")
print("  columna 1: |2| (renglón 1), |1| (renglón 2), |4| (renglón 3) -> gana el 3")
print("  columna 2: quedan 1 y 2; |7| contra |-3|                     -> gana el 1")
print("  columna 3: queda el 2")
print("  orden final: renglón 3, renglón 1, renglón 2")

A_ord = [A_des[2], A_des[0], A_des[1]]
b_ord = [b_des[2], b_des[0], b_des[1]]
print("\nSistema reordenado:")
for i in range(3):
    resto = sum(abs(A_ord[i][j]) for j in range(3) if j != i)
    print(f"  {A_ord[i]} | {b_ord[i]}   ->  |{A_ord[i][i]}| > {resto} ?  "
          f"{abs(A_ord[i][i]) > resto}")
print("ahora los tres renglones sí son dominantes")

# --- y si se itera SIN reordenar: dos barridas escritas a mano ---
print("\nDos barridas de Gauss-Seidel SIN reordenar (arranque [0, 0, 0]):")
x1 = x2 = x3 = 0.0
n1 = (19 - 7 * x2 - 1 * x3) / 2
n2 = (31 - 1 * n1 - 12 * x3) / -3
n3 = (3 - 4 * n1 - 1 * n2) / -1
print(f"  barrida 1: [{n1:.3f}, {n2:.3f}, {n3:.3f}]")
x1, x2, x3 = n1, n2, n3
n1 = (19 - 7 * x2 - 1 * x3) / 2
n2 = (31 - 1 * n1 - 12 * x3) / -3
n3 = (3 - 4 * n1 - 1 * n2) / -1
print(f"  barrida 2: [{n1:.3f}, {n2:.3f}, {n3:.3f}]")
print("en vez de acercarse a [1, 2, 3], los números se van de largo:")
print("la diagonal no domina, así que nada garantiza la convergencia.")


## 7 · Una barrida, a mano

Una **barrida** (o iteración) es el paso que se repite. La diferencia entre los
dos métodos está en **qué valores usa cada incógnita**:

**Jacobi**: todo se calcula con el vector viejo.

$$x_i^{(k+1)} = \frac{1}{a_{ii}}\left(b_i - \sum_{j \neq i} a_{ij}\,x_j^{(k)}\right)$$

**Gauss-Seidel**: cada valor se usa en cuanto sale.

$$x_i^{(k+1)} = \frac{1}{a_{ii}}\left(b_i - \sum_{j<i} a_{ij}\,x_j^{(k+1)} - \sum_{j>i} a_{ij}\,x_j^{(k)}\right)$$

La celda siguiente hace **una sola barrida** de cada método con el mismo sistema y
el mismo arranque, para que se vea la diferencia. Las preguntas 9 y 10 te piden
hacer esa cuenta a mano, con datos distintos.

> ✍️ **La celda anterior no es una función.** Es la cuenta hecha con los números de este ejemplo. Generalizarla —con ciclos y para cualquier $n$— es lo que tienes que escribir tú.


In [ ]:
#@title ✍️ Una barrida a mano: Jacobi y Gauss-Seidel, con números
# El sistema del documento. NO hay funciones aquí: la barrida se escribe término
# por término, con los números sustituidos, que es justo lo que se pide en las
# preguntas 9 y 10 (y lo que después hay que generalizar en la 11 y la 12).
A_bar = [[4, 1, -1],
         [2, 7, 1],
         [1, -3, 12]]
b_bar = [3, 19, 31]
x1, x2, x3 = 1.0, 1.0, 1.0        # el vector viejo, x⁰
def fmt(v):
    return f"{float(v):.6g}"


print("x⁰ =", [x1, x2, x3], "   (la solución del sistema es [1, 2, 3])")

print("\n--- JACOBI: las tres cuentas usan x⁰ ---")
j1 = (3 - 1 * x2 - (-1) * x3) / 4
j2 = (19 - 2 * x1 - 1 * x3) / 7
j3 = (31 - 1 * x1 - (-3) * x2) / 12
print(f"  x1 = (3  - 1·({x2}) - (-1)·({x3})) / 4   = {fmt(j1)}")
print(f"  x2 = (19 - 2·({x1}) - 1·({x3})) / 7     = {fmt(j2)}")
print(f"  x3 = (31 - 1·({x1}) - (-3)·({x2})) / 12 = {fmt(j3)}")
print("  x¹ (Jacobi) =", "[" + ", ".join(fmt(v) for v in (j1, j2, j3)) + "]")

print("\n--- GAUSS-SEIDEL: en cuanto sale un valor, se usa ---")
s1 = (3 - 1 * x2 - (-1) * x3) / 4          # igual que en Jacobi: es la primera
s2 = (19 - 2 * s1 - 1 * x3) / 7            # usa s1 NUEVO, no x1
s3 = (31 - 1 * s1 - (-3) * s2) / 12        # usa s1 y s2 NUEVOS
print(f"  x1 = (3  - 1·({x2}) - (-1)·({x3})) / 4   = {fmt(s1)}   igual que Jacobi")
print(f"  x2 = (19 - 2·({fmt(s1)}) - 1·({x3})) / 7     = {fmt(s2)}   ya usa el x1 nuevo")
print(f"  x3 = (31 - 1·({fmt(s1)}) - (-3)·({fmt(s2)})) / 12 = {fmt(s3)}")
print("  x¹ (Gauss-Seidel) =", "[" + ", ".join(fmt(v) for v in (s1, s2, s3)) + "]")

print("\nMismas cuentas y mismo arranque: Gauss-Seidel se acercó más a [1, 2, 3]")
print("porque reutilizó lo que iba calculando, sin esperar a la barrida siguiente.")


In [ ]:
#@title ✍️ Preguntas 9 y 10 · una barrida a mano, con tus datos
# Son cuentas: se contestan escribiendo el vector, no programando.
pregunta(9)
pregunta(10)


## 8 · Jacobi completo

Ahora el método entero. La función tiene que hacer **tres cosas**, en este orden:

1. **Reordenar** las filas para lograr la dominancia (regla de la sección 6).
2. **Iterar** con Jacobi.
3. **Parar** cuando el error baje de la tolerancia, o al llegar al máximo de
   iteraciones.

### El criterio de paro

Se compara la barrida nueva contra la anterior con la **norma infinito del
vector**:

$$\varepsilon = \frac{\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\|_\infty}{\|\mathbf{x}^{(k+1)}\|_\infty} = \frac{\max_i \left| x_i^{(k+1)} - x_i^{(k)} \right|}{\max_i \left| x_i^{(k+1)} \right|}$$

y se para cuando $\varepsilon < \text{tol}$.

> ⚠️ **No** se divide componente por componente. Si una incógnita vale cero (o se
> acerca a cero), el cociente $|\Delta x_i|/|x_i|$ se dispara y el método nunca
> cumple la tolerancia, aunque la solución ya esté bien. Con la norma el
> denominador es la componente **mayor**, que solo se anula si todo el vector es
> cero.

La convención va **declarada en la firma** y es la misma con la que se califica:

```
mi_jacobi(A, b, tol=1e-6, max_iter=100)   ->   return x, iteraciones
```

`x` se devuelve como **columna** (lista de listas). El segundo valor es cuántas
barridas hiciste: sirve para comprobar que respetaste `max_iter`.

> 📥 **Cómo llegan los datos.** `A` es una **lista de listas** y `b` una lista plana (por ejemplo `[[4, 1, -1], [2, 7, 1], [1, -3, 12]]` y `[3, 19, 31]`), no una matriz de `M.sym`. Así puedes recorrerlas con `for fila in A` y trabajar con `float` de forma natural.


In [ ]:
#@title ✍️ Escribe Jacobi completo
def mi_jacobi(A, b, tol=1e-6, max_iter=100):
    """Reordena, itera con Jacobi y devuelve (x como columna, iteraciones)."""
    raise NotImplementedError("Sigue los tres pasos de la sección 8")


practica('prog_jacobi')


In [ ]:
#@title 📝 Pregunta 11 (vale 1 punto)
pregunta(11)


## 9 · Gauss-Seidel completo

Misma estructura, un solo cambio en el cuerpo: **no se guarda el vector viejo**
para las cuentas, sino sólo para medir el error.

```
para k = 1, ..., max_iter:
    viejo = copia de x                 # sólo para el error
    para i = 1, ..., n:
        suma = (lo que aportan las demás incógnitas, con x YA actualizado)
        x[i] = (b[i] - suma) / A[i][i]
    si error(x, viejo) < tol:  devolver x, k
devolver x, max_iter
```

Como reutiliza lo recién calculado, suele necesitar menos iteraciones que Jacobi
para la misma tolerancia. En los sistemas de las preguntas, Jacobi tarda
alrededor del doble.

> 💡 Ojo con el orden: hay que copiar `x` **antes** de modificarlo, o el error
> sale siempre cero.

> 📥 **Cómo llegan los datos.** `A` es una **lista de listas** y `b` una lista plana (por ejemplo `[[4, 1, -1], [2, 7, 1], [1, -3, 12]]` y `[3, 19, 31]`), no una matriz de `M.sym`. Así puedes recorrerlas con `for fila in A` y trabajar con `float` de forma natural.


In [ ]:
#@title ✍️ Escribe Gauss-Seidel completo
def mi_gauss_seidel(A, b, tol=1e-6, max_iter=100):
    """Reordena, itera con Gauss-Seidel y devuelve (x como columna, iteraciones)."""
    raise NotImplementedError("Sigue el pseudocódigo de la sección 9")


practica('prog_seidel')


In [ ]:
#@title 📝 Pregunta 12 (vale 1 punto)
pregunta(12)


## 10 · Preguntas de teoría (1 a 8)

Las cinco primeras son de opción múltiple. La 6 pide una matriz (armar la
tridiagonal), la 7 un pivote modificado de Thomas y la 8 una entrada de la
diagonal de Cholesky.


In [ ]:
#@title 🧠 Preguntas 1 a 5 · opción múltiple
pregunta(1)
pregunta(2)
pregunta(3)
pregunta(4)
pregunta(5)


In [ ]:
#@title ✍️ Preguntas 6 a 8 · con tus matrices
# La 6 es una matriz (escríbela como [[1,2,3,4],[5,6,7,8],...]);
# la 7 y la 8 son un solo número.
pregunta(6)
pregunta(7)
pregunta(8)


## 11 · Calificar y enviar

- `calificar()` **no envía nada**: úsala las veces que quieras mientras trabajas.
  En las de programar te dice **qué caso** falló.
- `enviar(alumno_id)` sí manda la calificación y **gasta uno de tus 5 intentos**.


In [ ]:
#@title 📝 Calificar (sin enviar) — úsala las veces que quieras
calificar()


In [ ]:
#@title 📤 Enviar la calificación (máximo 5 intentos)
enviar(alumno_id)


In [ ]:
#@title 🔎 Comprobar lo que ya está guardado (no envía nada)
consultar_calificacion()


---
### ✅ Con esto cierras la Unidad 3

Ya tienes los cuatro pilares: Gauss con pivoteo, LU, los métodos iterativos y los
algoritmos para matrices con estructura. La comparación entre todos —cuál
conviene en cada caso— está en
[`05_comparativa.md`](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/master/docs/metodos/05_comparativa.md).

📤 Antes de cerrar: ejecuta `calificar()` y luego `enviar(alumno_id)`. Recuerda
que tienes **5 intentos**.
